In [7]:
import copy

In [ ]:
class SecondTick:
    HIGH_AMOUNT = 10000000
    def __init__(self, code):
        self.code = code
        self.reset() 

    def reset(self):
        self.time = None
        self.open = 0
        self.close = 0
        self.high = 0
        self.low = 0
        self.volume = 0
        self.amount = 0
        self.count = 0
        self.high_amount_count = 0

    def __str__(self):
        return ('code:' + self.code + '  ' +
                'time:' + str(self.time) + '  ' +
                'open:' + str(self.open) + '  ' +
                'close:' + str(self.close) + '  ' +
                'high:' + str(self.high) + '  ' +
                'low:' + str(self.low) + '  ' +
                'volume:' + str(self.volume) + '  ' +
                'amount:' + str(self.amount) + '  ' +
                'c:' + str(self.count) + '  ' +
                'hc:' + str(self.high_amount_count))

    def to_dict(self):
        return {'code': self.code,
                'time': self.time,
                'open': self.open,
                'close': self.close,
                'high': self.high,
                'low': self.low,
                'volume': self.volume,
                'amount': self.amount,
                'c': self.count,
                'hc': self.high_amount_count}

    def add_volume(self, v):
        self.volume += v

    def set_open(self, p):
        if self.open == 0:
            self.open = p

    def set_close(self, p):
        self.close = p

    def set_high(self, p):
        if p > self.high:
            self.high = p

    def set_low(self, p):
        if self.low == 0 or p < self.low:
            self.low = p

    def add_amount(self, a):
        self.amount += a
        self.count += 1
        if a > SecondTick.HIGH_AMOUNT:
            self.high_amount_count += 1

    def is_empty(self):
        return self.open == 0

    def finish(self, t):
        self.time = t
        c = copy.copy(self)
        self.reset()
        return c

In [ ]:
class TickTracker:
    def __init__(self, code, yesterday_close, callback):
        self.code = code
        self.tick_time = None
        self.search_date = search_date
        self.second_tick = SecondTick(code)
        self.second_ticks = []
        self.yesterday_close = yesterday_close
        self.open = 0
        self.is_skip = False
        self.callback = callback
        self.high = 0
        self.low = 0
        self.trend_time = None

    def push_second_tick(self, tick_time, ask_price):
        if self.second_tick.is_empty():
            return

        self.second_ticks.append(self.second_tick.finish(tick_time))
        self.tick_time = tick_time

        if len(self.second_ticks) >= 300 and ask_price > 0:   
            if self.trend_time is None or tick_time - self.trend_time > timedelta(seconds=10):
                self.callback(self.second_ticks, tick_time, ask_price)
                self.trend_time = tick_time

    def set_prices(self, price):
        if self.open == 0:
            self.open = price
            if self.open < self.yesterday_close:
                self.is_skip = True 

        if self.low == 0 or self.low < price:
            self.low = price

        if price > self.high:
            self.high = price

    def handle_tick(self, t):
        if self.tick_time is None:
            self.tick_time = t['date']

        if t['market_type'] == 50:
            self.set_prices(t['current_price'])

            if t['date'] - self.tick_time >= timedelta(seconds=1):
                self.push_second_tick(t['date'], t['ask_price'])
            self.second_tick.set_open(t['bid_price'])
            self.second_tick.set_close(t['bid_price'])
            self.second_tick.add_volume(t['volume'])
            self.second_tick.add_amount(t['volume'] * t['current_price'])
            self.second_tick.set_high(t['bid_price'])
            self.second_tick.set_low(t['bid_price'])
        else:
            self.push_second_tick(t['date'], 0)
